## Well-Guided Synthetic Samples 井先验合成样本展示

这个 notebook 用当前 `WellGuidedSyntheticDepthTraceDataset` 直接生成训练会看到的 synthetic depth trace。重点不是训练网络，而是检查合成数据本身是否合理：

- synthetic seismic 的振幅是否接近真实输入；
- residual / reflectivity 是否只出现在有效 taper 支撑区；
- `well_patch` 和 `unresolved_cluster` 是否真的产生了不同类型的高频结构；
- 多个高频事件是否经常被压进一个较宽的地震响应里。


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from ginn.enhance import load_well_resolution_prior_npz
from ginn_depth.config import DepthGINNConfig
from ginn_depth.data import build_dataset
from ginn_depth.physics import DepthForwardModel
from ginn_depth.synthetic import WellGuidedSyntheticDepthTraceDataset

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

SEED = 20260507
np.random.seed(SEED)
torch.manual_seed(SEED)

## 1) 配置路径

默认使用当前 depth GINN 的训练配置。notebook 会直接读取 `experiments/ginn_depth/train.yaml`。


In [ ]:
CONFIG_PATH = PROJECT_ROOT / "experiments" / "ginn_depth" / "train.yaml"
N_SYNTHETIC_EXAMPLES = 256

print("config:", CONFIG_PATH)
print("config exists:", CONFIG_PATH.exists())

In [ ]:
cfg = DepthGINNConfig.from_yaml(CONFIG_PATH, base_dir=PROJECT_ROOT)

# Showcase-only runtime overrides. The config schema and synthetic parameters still come from train.yaml.
cfg.synthetic_traces_per_epoch = N_SYNTHETIC_EXAMPLES
cfg.synthetic_batch_size = 16
cfg.device = "cpu"
cfg.num_workers = 0
cfg.pin_memory = False

print("prior:", cfg.resolution_prior_file)
print("prior exists:", cfg.resolution_prior_file.exists() if cfg.resolution_prior_file is not None else False)
cfg

## 2) 加载真实训练集和井先验

`build_dataset(cfg)` 会加载真实地震、LFM、Vp、mask、dynamic gain 和子波。synthetic dataset 会复用这些真实 trace 的背景和输入通道。


In [ ]:
prior = load_well_resolution_prior_npz(cfg.resolution_prior_file)
print("prior wells:", prior.n_wells)
print("prior samples:", prior.n_sample)
print("prior valid samples:", int(prior.well_mask.sum()))
print("prior residual summary:", prior.summary.get("residual", {}))

In [ ]:
dataset_bundle = build_dataset(cfg)
train_dataset = dataset_bundle.train_dataset

forward_model = DepthForwardModel(
    dataset_bundle.wavelet_time_s,
    dataset_bundle.wavelet_amp,
    depth_axis_m=dataset_bundle.depth_axis_m,
    amplitude_threshold=cfg.wavelet_amplitude_threshold,
).cpu()

synthetic_dataset = WellGuidedSyntheticDepthTraceDataset(
    train_dataset,
    prior,
    forward_model,
    num_examples=cfg.synthetic_traces_per_epoch,
    ai_min=cfg.ai_min,
    ai_max=cfg.ai_max,
    patch_fraction=cfg.synthetic_patch_fraction,
    unresolved_fraction=cfg.synthetic_unresolved_fraction,
    well_patch_scale_min=cfg.synthetic_well_patch_scale_min,
    well_patch_scale_max=cfg.synthetic_well_patch_scale_max,
    cluster_min_events=cfg.synthetic_cluster_min_events,
    cluster_max_events=cfg.synthetic_cluster_max_events,
    cluster_amp_abs_p95_min=cfg.synthetic_cluster_amp_abs_p95_min,
    cluster_amp_abs_p99_max=cfg.synthetic_cluster_amp_abs_p99_max,
    cluster_main_lobe_samples=cfg.synthetic_cluster_main_lobe_samples,
    residual_highpass_samples=cfg.synthetic_residual_highpass_samples,
    seismic_rms_match=cfg.synthetic_seismic_rms_match,
    seismic_rms_target=cfg.synthetic_seismic_rms_target,
    quality_gate_enabled=cfg.synthetic_quality_gate_enabled,
    max_residual_near_clip_fraction=cfg.synthetic_max_residual_near_clip_fraction,
    max_seismic_rms_ratio=cfg.synthetic_max_seismic_rms_ratio,
    max_seismic_abs_p99_ratio=cfg.synthetic_max_seismic_abs_p99_ratio,
    max_resample_attempts=cfg.synthetic_max_resample_attempts,
)

print("train traces:", len(train_dataset))
print("synthetic examples:", len(synthetic_dataset))
print("depth samples:", dataset_bundle.depth_axis_m.size)
print("train seismic RMS:", train_dataset.seis_rms)
print("train LFM scale:", train_dataset.lfm_scale)

## 3) 单条 synthetic trace

重复运行下面的 cell 可以随机刷新样本。`mode=0` 是井 residual patch，`mode=1` 是 unresolved high-frequency cluster。


In [ ]:
def as_1d(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().numpy().squeeze()
    return np.asarray(value).squeeze()


def finite_core(values, mask):
    values = np.asarray(values).reshape(-1)
    mask = np.asarray(mask).reshape(-1).astype(bool)
    valid = mask & np.isfinite(values)
    return values[valid]


def plot_synthetic_sample(sample, *, title="Synthetic sample"):
    depth = dataset_bundle.depth_axis_m
    target_seismic = as_1d(sample["target_seismic"])
    target_seismic_raw = as_1d(sample["target_seismic_raw"])
    real_obs = as_1d(sample["obs"])
    lfm = as_1d(sample["lfm_raw"])
    ai = as_1d(sample["target_ai"])
    residual = as_1d(sample["target_residual"])
    reflectivity = as_1d(sample["raw_reflectivity"])
    taper = as_1d(sample["taper_weight"])
    mask = as_1d(sample["loss_mask"]).astype(bool)
    mode = int(sample["synthetic_mode"].item())
    mode_name = "well_patch" if mode == 0 else "unresolved_cluster"

    fig, axes = plt.subplots(5, 1, figsize=(11, 9), sharex=True)
    axes[0].plot(depth, real_obs, color="0.65", lw=1.0, label="real normalized seismic (source trace)")
    axes[0].plot(depth, target_seismic_raw, color="tab:orange", lw=0.9, alpha=0.7, label="synthetic raw forward")
    axes[0].plot(depth, target_seismic, color="tab:blue", lw=1.2, label="synthetic RMS-matched")
    axes[0].legend(loc="upper right", fontsize=8)
    axes[0].set_ylabel("seismic")

    axes[1].plot(depth, lfm, color="0.4", lw=1.0, label="AI LFM")
    axes[1].plot(depth, ai, color="tab:red", lw=1.0, label="target AI")
    axes[1].legend(loc="upper right", fontsize=8)
    axes[1].set_ylabel("AI")

    axes[2].plot(depth, residual, color="tab:purple", lw=1.0)
    axes[2].axhline(0.0, color="0.2", lw=0.7)
    axes[2].set_ylabel("logAI residual")

    refl_depth = depth[:-1] if depth.size == reflectivity.size + 1 else np.arange(reflectivity.size)
    axes[3].plot(refl_depth, reflectivity, color="tab:green", lw=0.8)
    axes[3].axhline(0.0, color="0.2", lw=0.7)
    axes[3].set_ylabel("reflectivity")

    axes[4].plot(depth, taper, color="tab:brown", lw=1.2, label="taper")
    axes[4].fill_between(depth, 0.0, mask.astype(float), color="tab:cyan", alpha=0.25, label="loss mask")
    axes[4].set_ylim(-0.05, 1.05)
    axes[4].set_ylabel("support")
    axes[4].set_xlabel("depth / m")
    axes[4].legend(loc="upper right", fontsize=8)

    rms = np.sqrt(np.mean(finite_core(target_seismic, mask) ** 2))
    res_abs_p99 = np.percentile(np.abs(finite_core(residual, mask)), 99)
    fig.suptitle(f"{title} | mode={mode_name} | synthetic RMS={rms:.3f} | residual abs p99={res_abs_p99:.3f}")
    fig.tight_layout()
    return fig


sample = synthetic_dataset[0]
plot_synthetic_sample(sample, title="Random well-guided synthetic trace");

## 4) 对比两种 synthetic mode

这里强制各抽几条 `well_patch` 和 `unresolved_cluster`。重点看：cluster 是否把多个 residual / reflectivity 事件压在一个较短窗口里，而地震响应更像一个复合宽波形。


In [ ]:
def sample_mode(mode_code: int, n: int, *, max_tries: int = 2000):
    samples = []
    for _ in range(max_tries):
        sample = synthetic_dataset[0]
        if int(sample["synthetic_mode"].item()) == mode_code:
            samples.append(sample)
            if len(samples) >= n:
                return samples
    raise RuntimeError(f"Could not collect {n} samples for mode={mode_code}.")


patch_samples = sample_mode(0, 3)
cluster_samples = sample_mode(1, 3)


def plot_mode_grid(samples, *, title):
    depth = dataset_bundle.depth_axis_m
    fig, axes = plt.subplots(len(samples), 3, figsize=(13, 2.2 * len(samples)), sharex="col")
    if len(samples) == 1:
        axes = axes[np.newaxis, :]
    for row, sample in enumerate(samples):
        seismic = as_1d(sample["target_seismic"])
        residual = as_1d(sample["target_residual"])
        reflectivity = as_1d(sample["raw_reflectivity"])
        mask = as_1d(sample["loss_mask"]).astype(bool)
        axes[row, 0].plot(depth, seismic, color="tab:blue", lw=1.0)
        axes[row, 0].fill_between(depth, seismic.min(), seismic.max(), where=mask, color="0.85", alpha=0.35)
        axes[row, 1].plot(depth, residual, color="tab:purple", lw=1.0)
        axes[row, 1].axhline(0.0, color="0.2", lw=0.6)
        refl_depth = depth[:-1] if depth.size == reflectivity.size + 1 else np.arange(reflectivity.size)
        axes[row, 2].plot(refl_depth, reflectivity, color="tab:green", lw=0.8)
        axes[row, 2].axhline(0.0, color="0.2", lw=0.6)
        axes[row, 0].set_ylabel(f"#{row + 1}")
    axes[0, 0].set_title("target seismic")
    axes[0, 1].set_title("target logAI residual")
    axes[0, 2].set_title("target reflectivity")
    for ax in axes[-1, :]:
        ax.set_xlabel("depth / m")
    fig.suptitle(title)
    fig.tight_layout()
    return fig


plot_mode_grid(patch_samples, title="well_patch synthetic examples")
plot_mode_grid(cluster_samples, title="unresolved_cluster synthetic examples");

## 5) 批量统计：真实 vs synthetic 振幅

这一步很关键。如果 synthetic seismic 的 RMS / p99 明显偏离真实输入，网络会先学到一个错误域。


In [ ]:
def summarize_one_sample(sample):
    mask = as_1d(sample["loss_mask"]).astype(bool)
    real = finite_core(as_1d(sample["obs"]), mask)
    synth = finite_core(as_1d(sample["target_seismic"]), mask)
    residual = finite_core(as_1d(sample["target_residual"]), mask)
    ai = finite_core(as_1d(sample["target_ai"]), mask)
    mode = int(sample["synthetic_mode"].item())
    return {
        "mode": mode,
        "real_rms": float(np.sqrt(np.mean(real**2))) if real.size else np.nan,
        "synthetic_rms": float(np.sqrt(np.mean(synth**2))) if synth.size else np.nan,
        "real_abs_p99": float(np.percentile(np.abs(real), 99)) if real.size else np.nan,
        "synthetic_abs_p99": float(np.percentile(np.abs(synth), 99)) if synth.size else np.nan,
        "residual_rms": float(np.sqrt(np.mean(residual**2))) if residual.size else np.nan,
        "residual_abs_p99": float(np.percentile(np.abs(residual), 99)) if residual.size else np.nan,
        "ai_low_hit": float(np.mean(ai <= cfg.ai_min * 1.0001)) if ai.size else np.nan,
        "ai_high_hit": float(np.mean(ai >= cfg.ai_max * 0.9999)) if ai.size else np.nan,
        "rms_scale": float(sample["synthetic_rms_scale"].item()),
    }


def collect_summaries(n: int = 256):
    rows = [summarize_one_sample(synthetic_dataset[i]) for i in range(n)]
    return {key: np.asarray([row[key] for row in rows]) for key in rows[0]}


stats = collect_summaries(N_SYNTHETIC_EXAMPLES)
print(
    "mode counts: well_patch=", int(np.sum(stats["mode"] == 0)), "unresolved_cluster=", int(np.sum(stats["mode"] == 1))
)
for key in [
    "real_rms",
    "synthetic_rms",
    "real_abs_p99",
    "synthetic_abs_p99",
    "residual_rms",
    "residual_abs_p99",
    "rms_scale",
]:
    values = stats[key]
    print(
        f"{key:20s} mean={np.nanmean(values):.4f} p50={np.nanpercentile(values, 50):.4f} p95={np.nanpercentile(values, 95):.4f}"
    )
print("AI bound hit mean: low=", np.nanmean(stats["ai_low_hit"]), "high=", np.nanmean(stats["ai_high_hit"]))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 7))

axes[0, 0].hist(stats["real_rms"], bins=30, alpha=0.55, label="real source")
axes[0, 0].hist(stats["synthetic_rms"], bins=30, alpha=0.55, label="synthetic")
axes[0, 0].set_title("core RMS")
axes[0, 0].legend()

axes[0, 1].hist(stats["real_abs_p99"], bins=30, alpha=0.55, label="real source")
axes[0, 1].hist(stats["synthetic_abs_p99"], bins=30, alpha=0.55, label="synthetic")
axes[0, 1].set_title("abs p99")
axes[0, 1].legend()

axes[0, 2].scatter(stats["real_rms"], stats["synthetic_rms"], s=14, alpha=0.55)
lim = [0, max(np.nanmax(stats["real_rms"]), np.nanmax(stats["synthetic_rms"])) * 1.05]
axes[0, 2].plot(lim, lim, color="0.2", lw=1.0)
axes[0, 2].set_xlim(lim)
axes[0, 2].set_ylim(lim)
axes[0, 2].set_title("synthetic RMS vs real RMS")
axes[0, 2].set_xlabel("real")
axes[0, 2].set_ylabel("synthetic")

axes[1, 0].hist(stats["residual_rms"], bins=30, color="tab:purple", alpha=0.7)
axes[1, 0].set_title("target residual RMS")

axes[1, 1].hist(stats["residual_abs_p99"], bins=30, color="tab:purple", alpha=0.7)
axes[1, 1].set_title("target residual abs p99")

axes[1, 2].hist(stats["rms_scale"], bins=30, color="tab:orange", alpha=0.7)
axes[1, 2].set_title("synthetic seismic RMS scale")

fig.suptitle("Batch synthetic diagnostics")
fig.tight_layout()

## 6) 频谱对比

这里不要求 synthetic residual 逐点等于某口井，但希望 synthetic residual 的高频能量规模接近井先验，同时 synthetic seismic 的带宽不要脱离真实输入。


In [ ]:
def normalized_amp_spectrum(values):
    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]
    if values.size < 8:
        return None, None
    centered = values - values.mean()
    amp = np.abs(np.fft.rfft(centered))
    freq = np.fft.rfftfreq(centered.size, d=1.0)
    if amp.max() > 0:
        amp = amp / amp.max()
    return freq, amp


def mean_spectrum(curves, freq_out=np.linspace(0.0, 0.5, 128)):
    spectra = []
    for values in curves:
        freq, amp = normalized_amp_spectrum(values)
        if freq is not None:
            spectra.append(np.interp(freq_out, freq, amp))
    if not spectra:
        return freq_out, np.full_like(freq_out, np.nan)
    return freq_out, np.nanmean(np.stack(spectra), axis=0)


real_seismic_curves = []
synthetic_seismic_curves = []
synthetic_residual_curves = []
for _ in range(128):
    sample = synthetic_dataset[0]
    mask = as_1d(sample["loss_mask"]).astype(bool)
    real_seismic_curves.append(finite_core(as_1d(sample["obs"]), mask))
    synthetic_seismic_curves.append(finite_core(as_1d(sample["target_seismic"]), mask))
    synthetic_residual_curves.append(finite_core(as_1d(sample["target_residual"]), mask))

prior_residual_curves = []
for row in range(prior.n_wells):
    row_mask = prior.well_mask[row]
    if int(row_mask.sum()) >= 8:
        prior_residual_curves.append(prior.residual_log_ai[row, row_mask])

freq, real_spec = mean_spectrum(real_seismic_curves)
_, synth_spec = mean_spectrum(synthetic_seismic_curves, freq)
_, synth_res_spec = mean_spectrum(synthetic_residual_curves, freq)
_, prior_res_spec = mean_spectrum(prior_residual_curves, freq)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(freq, real_spec, label="real source seismic", lw=1.4)
axes[0].plot(freq, synth_spec, label="synthetic seismic", lw=1.4)
axes[0].set_title("normalized seismic amplitude spectrum")
axes[0].set_xlabel("cycles / sample")
axes[0].set_ylabel("normalized amplitude")
axes[0].legend()

axes[1].plot(freq, prior_res_spec, label="well prior residual", lw=1.4)
axes[1].plot(freq, synth_res_spec, label="synthetic residual", lw=1.4)
axes[1].set_title("normalized residual amplitude spectrum")
axes[1].set_xlabel("cycles / sample")
axes[1].legend()

fig.tight_layout()

## 7) 只看调谐窗口

下面从 `unresolved_cluster` 中自动找 residual 绝对值最大的窗口。理想情况下，窗口内会有多个高频 residual / reflectivity 事件，但 seismic 表现为更平滑、更带限的复合响应。


In [ ]:
cluster = sample_mode(1, 1)[0]
depth = dataset_bundle.depth_axis_m
residual = as_1d(cluster["target_residual"])
seismic = as_1d(cluster["target_seismic"])
reflectivity = as_1d(cluster["raw_reflectivity"])
lfm = as_1d(cluster["lfm_raw"])
ai = as_1d(cluster["target_ai"])
mask = as_1d(cluster["loss_mask"]).astype(bool)

active = np.flatnonzero(mask)
center = active[np.argmax(np.abs(residual[active]))] if active.size else int(np.argmax(np.abs(residual)))
half = max(12, int(cfg.synthetic_cluster_main_lobe_samples or 12) * 3)
start = max(0, center - half)
stop = min(depth.size, center + half + 1)
sl = slice(start, stop)
refl_sl = slice(start, min(stop, reflectivity.size))

fig, axes = plt.subplots(4, 1, figsize=(10, 7), sharex=True)
axes[0].plot(depth[sl], seismic[sl], color="tab:blue", lw=1.4)
axes[0].set_ylabel("seismic")
axes[1].plot(depth[sl], residual[sl], color="tab:purple", marker=".", lw=1.0)
axes[1].axhline(0.0, color="0.2", lw=0.7)
axes[1].set_ylabel("residual")
axes[2].plot(depth[refl_sl], reflectivity[refl_sl], color="tab:green", marker=".", lw=0.9)
axes[2].axhline(0.0, color="0.2", lw=0.7)
axes[2].set_ylabel("reflectivity")
axes[3].plot(depth[sl], lfm[sl], color="0.45", label="AI LFM")
axes[3].plot(depth[sl], ai[sl], color="tab:red", label="target AI")
axes[3].set_ylabel("AI")
axes[3].set_xlabel("depth / m")
axes[3].legend(fontsize=8)

fig.suptitle("Unresolved cluster zoom window")
fig.tight_layout()